In [ ]:
import os
from dotenv import load_dotenv
from autogen import AssistantAgent, UserProxyAgent
load_dotenv()

# llm_config = {
#     "config_list": [
#         {
#             "model": "gpt-oss:20b-cloud",
#             #"model_name": "gpt-oss",
#             "temperature": 0.7,
#             "base_url": "http://localhost:11434/v1",
#             "api_key": "ollama"

#         }
#     ]
# }

llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY2"), # Paste your key here
        }
    ]
}

llm_configForManager = {
    "config_list": [
        {
            "model": "gpt-oss:20b-cloud",
           #"model_name": "gpt-oss",
            "temperature": 0.7,
            "base_url": "http://localhost:11434/v1",
            "api_key": "ollama"
        }
    ]
}

### Read log files 

In [2]:
import os
def Read_logs():
    folder = "./logs"
    logs = []

    if not os.path.exists(folder):
        print("Log folder does not exist.")
        return logs

    for filename in os.listdir(folder):
        with open(os.path.join(folder, filename), 'r') as file :
            logs += f"\n--- Log from {file} ---\n{file.read()}"  
    return logs[:3000] if logs else "No logs found."

Agent 

In [3]:
log_analyst = AssistantAgent(
    name="LogAnalyst",
    description="An assistant that analyzes system logs and provides insights.",
    system_message="""

            You are a helpful log analyst who can read and interpret system logs.
            You will be provided with system logs, and your task is to identify any errors, warnings, or notable events.
            You will summarize your findings and suggest possible actions or improvements based on the log data.

            End With : 'analysis complete.'
""",
    llm_config=llm_config
)

Agent proxy

In [6]:
agent_proxy = UserProxyAgent(
    name="LoggingProxy",
    description="A proxy agent that provides system logs to the LogAnalyst assistant.",
    system_message="""
            You are a proxy agent that provides system logs to the LogAnalyst assistant.
            When the LogAnalyst requests logs, you will supply them.
            Ensure that the logs are formatted correctly and are easy to read.
            Always end your response with 'logs provided.'
        """,
    code_execution_config=False,
    human_input_mode="NEVER"
)


In [7]:
from autogen import GroupChat, GroupChatManager


logs = Read_logs()

##Team collaboration
chat = GroupChat(
                agents=[
                    log_analyst, 
                    agent_proxy
                    ],
                 max_round=6,
                 speaker_selection_method="round_robin"
                 )
manager= GroupChatManager(groupchat=chat,llm_config=llm_configForManager)

## Start conversation. 
message=f"""
    Provide the system logs for analysis:
    {logs}
identify any errors, warnings, or notable events. and summarize your findings and suggest possible actions or improvements based on the log data.
"""
agent_proxy.initiate_chat(manager, message=message)



LoggingProxy (to chat_manager):


    Provide the system logs for analysis:
    ['\n', '-', '-', '-', ' ', 'L', 'o', 'g', ' ', 'f', 'r', 'o', 'm', ' ', '<', '_', 'i', 'o', '.', 'T', 'e', 'x', 't', 'I', 'O', 'W', 'r', 'a', 'p', 'p', 'e', 'r', ' ', 'n', 'a', 'm', 'e', '=', "'", '.', '/', 'l', 'o', 'g', 's', '/', 'l', 'o', 'g', 's', '1', '.', 'l', 'o', 'g', 's', "'", ' ', 'm', 'o', 'd', 'e', '=', "'", 'r', "'", ' ', 'e', 'n', 'c', 'o', 'd', 'i', 'n', 'g', '=', "'", 'U', 'T', 'F', '-', '8', "'", '>', ' ', '-', '-', '-', '\n', '2', '0', '2', '5', '-', '1', '2', '-', '1', '4', ' ', '1', '0', ':', '0', '0', ':', '0', '1', ' ', '[', 'I', 'N', 'F', 'O', ']', ' ', ' ', '[', 'A', 'u', 't', 'h', 'S', 'e', 'r', 'v', 'i', 'c', 'e', ']', ' ', 'U', 's', 'e', 'r', ' ', 'l', 'o', 'g', 'i', 'n', ' ', 's', 'u', 'c', 'c', 'e', 's', 's', 'f', 'u', 'l', ' ', '|', ' ', 'u', 's', 'e', 'r', '_', 'i', 'd', '=', '1', '0', '0', '1', '\n', '2', '0', '2', '5', '-', '1', '2', '-', '1', '4', ' ', '1', '0', ':', '0', '

ChatResult(chat_id=144104825070868924822834618820607836824, chat_history=[{'content': '\n    Provide the system logs for analysis:\n    [\'\\n\', \'-\', \'-\', \'-\', \' \', \'L\', \'o\', \'g\', \' \', \'f\', \'r\', \'o\', \'m\', \' \', \'<\', \'_\', \'i\', \'o\', \'.\', \'T\', \'e\', \'x\', \'t\', \'I\', \'O\', \'W\', \'r\', \'a\', \'p\', \'p\', \'e\', \'r\', \' \', \'n\', \'a\', \'m\', \'e\', \'=\', "\'", \'.\', \'/\', \'l\', \'o\', \'g\', \'s\', \'/\', \'l\', \'o\', \'g\', \'s\', \'1\', \'.\', \'l\', \'o\', \'g\', \'s\', "\'", \' \', \'m\', \'o\', \'d\', \'e\', \'=\', "\'", \'r\', "\'", \' \', \'e\', \'n\', \'c\', \'o\', \'d\', \'i\', \'n\', \'g\', \'=\', "\'", \'U\', \'T\', \'F\', \'-\', \'8\', "\'", \'>\', \' \', \'-\', \'-\', \'-\', \'\\n\', \'2\', \'0\', \'2\', \'5\', \'-\', \'1\', \'2\', \'-\', \'1\', \'4\', \' \', \'1\', \'0\', \':\', \'0\', \'0\', \':\', \'0\', \'1\', \' \', \'[\', \'I\', \'N\', \'F\', \'O\', \']\', \' \', \' \', \'[\', \'A\', \'u\', \'t\', \'h\', \'S\', \'e\

# AutoGen Log Analysis Agent - Project Documentation

## 1. Project Overview
This project utilizes Microsoft's **AutoGen** framework to create a multi-agent system that automatically analyzes system log files. It demonstrates how to set up an **AssistantAgent** (the analyst) and a **UserProxyAgent** (the supplier of data) to collaborate on a task without human intervention.

---

## 2. Setup & Dependencies

### Prerequisites
* Python 3.8+
* OpenAI API Key

### Required Libraries
```bash
pip install pyautogen python-dotenv
```

### Environment Configuration (.env)
The project uses `python-dotenv` to securely manage API keys.
Create a file named `.env` in the root directory:
```env
OPENAI_API_KEY2=sk-proj-your-actual-api-key-here
```
*(Note: The code specifically looks for `OPENAI_API_KEY2` based on your implementation)*

---

## 3. Code Breakdown

### A. Imports and Environment Loading
```python
import os
from dotenv import load_dotenv
from autogen import AssistantAgent, UserProxyAgent

load_dotenv() # Loads variables from .env into os.environ
```
* **Purpose:** Securely loads the API key so it isn't hardcoded in the script.
* **Key Learnings:** Scripts do not load `.env` files automatically (unlike some UI frameworks); `load_dotenv()` is mandatory.

### B. LLM Configuration
```python
llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY2"),
        }
    ]
}
```
* **Purpose:** Defines which AI model the agents will use.
* **Why `gpt-4o-mini`?** It is cost-effective and sufficient for text analysis tasks like reading logs.

### C. Log Reading Function
**⚠️ Important Logic Note:**
```python
def Read_logs():
    folder = "./logs"
    logs = [] # <--- Potential Issue Here (See Section 4)
    # ... file reading logic ...
    # logs += ...
    return logs[:3000]
```
* **Purpose:** Scans the `./logs` directory and compiles content into a single variable.
* **Truncation:** Returns only the first 3000 characters to prevent overflowing the LLM's context window.

### D. Agent Definitions

#### 1. The Analyst (AssistantAgent)
```python
log_analyst = AssistantAgent(
    name="LogAnalyst",
    system_message="You are a helpful log analyst...",
    llm_config=llm_config
)
```
* **Role:** The "Brain". It receives text, processes it using the LLM, and generates the insight.
* **System Message:** Instructions on *how* to behave (identify errors, summarize, suggest actions).

#### 2. The Proxy (UserProxyAgent)
```python
agent_proxy = UserProxyAgent(
    name="LoggingProxy",
    human_input_mode="NEVER",
    code_execution_config=False
)
```
* **Role:** The "Trigger". In this specific setup, it acts as the entity initiating the request.
* **`human_input_mode="NEVER"`:** Ensures the script runs fully automatically without asking the user for permission at every step.

### E. Group Chat Orchestration
```python
chat = GroupChat(
    agents=[log_analyst, agent_proxy],
    max_round=5,
    speaker_selection_method="round_robin"
)
manager = GroupChatManager(groupchat=chat, llm_config=llm_config)
```
* **GroupChat:** Defines the "room" where agents talk.
* **Manager:** The "moderator" that decides whose turn it is to speak and passes messages to the LLM.

### F. Execution
```python
message = f"Provide the system logs for analysis: {logs}..."
agent_proxy.initiate_chat(manager, message=message)
```
* **Action:** The proxy sends the initial prompt (containing the raw logs) to the manager, kicking off the analysis loop.

---

## 4. Common Issues & Fixes

### Issue 1: "AuthenticationError" or "Invalid API Key"
* **Symptom:** The script crashes saying the API key is incorrect or missing.
* **Cause:** The script cannot read the `.env` file, or the `.env` file contains Python code (e.g., `os.getenv(...)`) instead of just the key.
* **Fix:**
    1. Ensure `.env` contains only `KEY_NAME=sk-1234...`.
    2. Ensure `load_dotenv()` is called at the very top of the script.

### Issue 2: Messy Log Output (List of Characters)
* **Symptom:** The AI receives data that looks like `['L', 'o', 'g', ' ', 'f', 'i', 'l', 'e'...]`.
* **Cause:** Initializing `logs = []` (list) and then using `+=` with a string. Python splits the string into characters when adding to a list.
* **Fix:** Initialize as a string.
    ```python
    # Incorrect
    logs = [] 
    
    # Correct
    logs = "" 
    ```

### Issue 3: Missing Log Folder
* **Symptom:** Script returns "No logs found" immediately.
* **Fix:** Create a folder named `logs` in the same directory as your script and add at least one `.log` or `.txt` file inside it.

---

## 5. Future Improvements
1.  **Dynamic Context:** Instead of truncating at 3000 characters, implement a "chunking" strategy to analyze larger log files in parts.